*We deleted the one part of a top agent's behaviour that a conservation map said was free to vary. The farm stopped growing. The failure was not in trading.*

**What this is.** A knockout experiment on a leaderboard, and what it says about the difference between *where an agent varies* and *where an agent may be changed*. Those are not the same set, and this notebook measures the gap on 1,067,040 turns of the ten highest-rated submissions.

**The result in one line.** Across all ten agents, a turn carries a sell order **37.9 %** of the time in general and **76.4 %** of the time when the agent buys land or an animal on that same turn. And when both appear, the sell is ordered **before** the purchase in **9,208 of 9,208** cases. Zero exceptions, ten independent teams.

**Why that matters to anyone replaying a recorded route.** The engine settles market orders in list order, so a sale placed before a purchase in the same list *pays for it inside the turn*. **A recorded sell schedule is therefore also a financing schedule**, and it is invisible if you read the stream as a sequence of trades. Delete it and a purchase silently fails.

**And the methodological finding, which is the point.** [The previous notebook](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) mapped which turns of a replay the source agent had been observed to vary. A conservation map, in the genomic sense. A conservation map tells you where variation is **tolerated**. This notebook is what happens when you read it as telling you where change is **safe**.

---

## What was already established

Four earlier notebooks set this up, each treating the ladder as a population rather than as a set of independent optimisers.

**There are species.** [The leaderboard is a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient): what you meet at one rating band is not what you meet at another.

**Much of the field shares an opening.** [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening): unrelated teams emit identical actions, turn for turn, deep into the episode.

**A route splits into a plan and a market channel, and the second one moves.** [Mutants at the top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay): three of the top ten hold one construction plan across every episode while varying their trading on up to 140 turns. That map is the object this notebook tests.

**And the field turns over by sweeps.** [The leaderboard has a fossil record](https://www.kaggle.com/code/destbreso/the-leaderboard-has-a-fossil-record): a lineage takes a third of all seats in two days and is extinct in five.

So the obvious next move was to build the mutant the map licenses. Keep the construction plan of a strong agent exactly as recorded, throw away its entire market channel, and generate the trades ourselves from our own inventory and the prices we can see. In genetics this is a **knockout**: remove a region the alignment says is not conserved, and see what the organism does.

In [ ]:
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})

# One palette for the series. Checked for monotone lightness so the three stay
# separable in greyscale and under the common colour-vision deficiencies.
SELL  = "#0F766E"   # the trading channel
INPUT = "#B45309"   # seeds and bought goods
STRUCT= "#0F172A"   # hire, land, animals: the channel that is really plan
NULL  = "#94A3B8"   # the shuffled baseline
ACC   = "#B91C1C"

def find_dir(marker, roots=("/kaggle/input", ".", "..", "data")):
    hits = []
    for r in roots:
        p = Path(r)
        if p.exists():
            hits += [q.parent for q in p.rglob(marker)]
    if not hits:
        raise SystemExit(f"could not find {marker}")
    return sorted(hits, key=lambda q: len(str(q)))[0]

DATA = find_dir("streams.parquet")
print("dataset:", DATA)

streams = pd.read_parquet(DATA / "streams.parquet")
episodes = pd.read_csv(DATA / "episodes.csv")
print(f"{len(streams):,} turns, {streams.episode_id.nunique():,} episodes, "
      f"{streams.submission.nunique()} submissions")

# Rank and team name, so the figures can be read without a lookup table.
RANK = (episodes.drop_duplicates("submission")
        .set_index("submission")[["rank", "team"]].to_dict("index")
        if {"rank", "team"} <= set(episodes.columns) else {})

def has_cjk(s):
    return any("⺀" <= ch <= "鿿" or "가" <= ch <= "힯" for ch in str(s))

def label(sub):
    meta = RANK.get(sub) or RANK.get(int(sub)) or {}
    r, t = meta.get("rank"), str(meta.get("team", sub))
    # Kaggle's images carry no CJK font, so a name that would render as tofu is
    # replaced by its submission id rather than by four empty boxes.
    if has_cjk(t) or not t or t == "nan":
        t = f"team {sub}"
    return f"{r}. {t[:18]}" if r else t[:20]

## 1. The knockout, and the phenotype

The agent under test replays the construction plan of a top-ten route exactly. Every `farmer` action, every `hands` action, on the same turn, forever, and generates every sell order itself from its own shed, its own bank and the quoted prices. Nothing recorded survives in that channel.

Before measuring anything we verified the control is bit-identical: with the new seller switched off, the child emits the same action dictionary as its parent on **4,314 turns across both seats, with zero differences**. That gate exists because a change and a bug are otherwise the same observation.

Then it played the same 40 recorded matchups the parent plays, at the same seeds, from the same seats, against the same recorded opponents.

**It lost all forty.** And the interesting part is not that it lost. It is *what* broke.

*One matched pair of episodes at the same seed and the same seat, in our own harness, written up in `docs/MARKET_EXECUTION.md` section 2. It is a single experiment, stated here rather than recomputed: nothing in this notebook replays those two episodes. The population-scale evidence is sections 3 and 4, and those ARE recomputed below, from the published corpus.*

| quantity | recorded route | knockout |
|---|---:|---:|
| quadrants unlocked | 3 | **2** |
| LOCKED tiles at the end | 25 | **50** |
| MILK units sold | 320 | **70** |
| board tiles differing from the parent | 0 | **32** |

In [ ]:
# The table above, as data. One definition, so the panels and the table cannot
# disagree; the highlight under the milk panel is computed from it.
PHENO = [('quadrants unlocked', 3, 2), ('LOCKED tiles at the end', 25, 50), ('MILK units sold', 320, 70), ('board tiles differing from the parent', 0, 32)]
P = {q: (a, b) for q, a, b in PHENO}
milk_lost = 1 - P["MILK units sold"][1] / P["MILK units sold"][0]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
for ax, (title, key, hi) in zip(axes, [
        ("LOCKED tiles", "LOCKED tiles at the end", "a whole quadrant, all season"),
        ("MILK units sold", "MILK units sold", f"{milk_lost:.0%} of the herd's output"),
        ("board tiles wrong", "board tiles differing from the parent",
         "the plan was replayed verbatim")]):
    a, b = P[key]
    ax.bar([0, 1], [a, b], color=[STRUCT, ACC], width=0.55)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["route", "knockout"])
    ax.set_title(title); ax.set_ylabel("")
    ax.text(0.5, max(a, b) * 1.02, hi, ha="center", va="bottom",
            fontsize=8, color="#475569")
    ax.set_ylim(0, max(a, b) * 1.28)
fig.suptitle("Deleting the trades broke the construction, not the trading",
             y=1.06, fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

**Read the third panel first.** The construction plan was replayed verbatim and the final farm still differs by 32 tiles of 100. That is not possible if the plan is the only thing that builds the farm.

Twenty-five of those tiles are one locked quadrant. The knockout never bought it.

## 2. The mechanism, which is in the engine rather than in the strategy

Here is the same turn in both runs. The recorded route is on top.

```
route     bank $598    market [SELL WOOL 9, SELL FERTILIZER 3, BUY_LAND]
knockout  bank $105    market [BUY_LAND]
```

Two properties of the engine make this decisive, and both are checkable in its source rather than inferred.

**Market orders settle in list order.** The interpreter walks the order list index by index, so a sale at index 0 has already credited the bank when the purchase at index 2 is attempted. The route's two sells raise about \$1,300, and the \$1,000 land purchase then succeeds *within the same turn*.

**A purchase that cannot afford itself is a silent no-op.** No error, no penalty, no signal in the action stream. The quadrant simply stays locked for the remaining 560 turns.

The knockout sold the same goods one turn later. That was enough.

**And there is a second, subtler reason it was late.** The engine applies every farmer and hands action *before* it processes the market. So an order executes against a shed that the turn's own pickups and drops have already changed. At the turn above, the recorded route sells nine `WOOL` out of a shed that holds **none** when the turn begins. The wool arrives during the turn, carried by a hand that drops it. An agent that reads its inventory literally therefore sells one turn late, every time.

Fixing only that, with nothing else changed, took the knockout from **0 of 20** to **7 of 20**.

## 3. Does the field do this on purpose? First test: when do they sell?

If the sells are financing, agents should sell **more often on the turns they buy**. If the ordering is a coincidence of style, the rate should not move.

Every turn of all ten submissions, from the published stream table.

In [ ]:
STRUCTURAL = {"BUY_LAND", "BUY_ANIMAL"}
PURCHASE   = {"BUY_LAND", "BUY_ANIMAL", "BUY_SEED", "BUY_PRODUCT", "HIRE"}

def ops_of(s):
    if not s or s == "null":
        return []
    try:
        o = json.loads(s)
    except Exception:
        return []
    return [x[0] for x in o if isinstance(x, list) and x] if isinstance(o, list) else []

rows = []
for sub, g in streams.groupby("submission"):
    n = ns = a = b = 0
    for s in g["market"]:
        ops = ops_of(s)
        n += 1
        sell = "SELL" in ops
        ns += sell
        if STRUCTURAL & set(ops):
            a += 1; b += sell
    if a:
        rows.append({"submission": sub, "turns": n, "p_sell": ns / n,
                     "struct_turns": a, "p_sell_given_struct": b / a,
                     "lift": (b / a) / (ns / n)})
cond = pd.DataFrame(rows).sort_values("lift", ascending=False).reset_index(drop=True)
cond["agent"] = cond.submission.map(label)
display(cond[["agent", "turns", "p_sell", "struct_turns",
              "p_sell_given_struct", "lift"]]
        .style.format({"turns": "{:,}", "struct_turns": "{:,}",
                       "p_sell": "{:.1%}", "p_sell_given_struct": "{:.1%}",
                       "lift": "{:.2f}x"}))

tot_turns = cond.turns.sum()
tot_sell  = (cond.p_sell * cond.turns).sum()
tot_st    = cond.struct_turns.sum()
tot_stsell= (cond.p_sell_given_struct * cond.struct_turns).sum()
print(f"\nALL TEN: {tot_sell/tot_turns:.1%} of {tot_turns:,} turns carry a sell, "
      f"against {tot_stsell/tot_st:.1%} of the {tot_st:,} turns that buy land or an animal")

In [ ]:
d = cond.sort_values("p_sell_given_struct")
y = np.arange(len(d))
fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.hlines(y, d.p_sell, d.p_sell_given_struct, color="#CBD5E1", lw=3, zorder=1)
ax.scatter(d.p_sell, y, s=70, color=NULL, zorder=3, label="any turn")
ax.scatter(d.p_sell_given_struct, y, s=70, color=SELL, zorder=3,
           label="a turn that buys land or an animal")
for yy, (lo, hi) in enumerate(zip(d.p_sell, d.p_sell_given_struct)):
    ax.text(hi + 0.015, yy, f"{hi/lo:.1f}x", va="center", fontsize=8.5,
            color=SELL, fontweight="bold")
ax.set_yticks(y); ax.set_yticklabels([label(s) for s in d.submission], fontsize=8.5)
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("share of turns carrying at least one SELL order")
ax.set_title("Every one of the ten sells about twice as often on the turns it buys",
             loc="left")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

**Every single one of the ten, without exception.** The pooled rate goes from 37.9 % to 76.4 %, a lift of 2.02x, and the smallest individual lift is 1.83x.

This is not a claim that they *intend* it. It is a claim that the behaviour is there, in all ten, at a size no sampling noise produces over a million turns.

## 4. Second test, and this is the one with no exceptions: what order?

Selling more often on a buying turn would still be consistent with mere correlation, busy turns are busy. The ordering is not.

For every turn that contains **both** a sell and a structural purchase, ask whether the sell is at a lower index. Against a null in which the same multiset of operations is shuffled.

In [ ]:
import random
random.seed(7)

rows = []
for sub, g in streams.groupby("submission"):
    n = first = null_n = null_hit = 0
    for s in g["market"]:
        ops = ops_of(s)
        if "SELL" not in ops:
            continue
        st = [i for i, o in enumerate(ops) if o in STRUCTURAL]
        if not st:
            continue
        sells = [i for i, o in enumerate(ops) if o == "SELL"]
        n += 1
        first += min(sells) < max(st)
        p = list(ops); random.shuffle(p)
        ps = [i for i, o in enumerate(p) if o == "SELL"]
        pb = [i for i, o in enumerate(p) if o in STRUCTURAL]
        if ps and pb:
            null_n += 1; null_hit += min(ps) < max(pb)
    if n:
        rows.append({"submission": sub, "turns": n, "sell_first": first / n,
                     "exceptions": n - first,
                     "shuffled": null_hit / null_n if null_n else np.nan})
order = pd.DataFrame(rows).sort_values("turns", ascending=False).reset_index(drop=True)
order["agent"] = order.submission.map(label)
display(order[["agent", "turns", "sell_first", "exceptions", "shuffled"]]
        .style.format({"turns": "{:,}", "sell_first": "{:.1%}",
                       "shuffled": "{:.1%}"}))
print(f"\nALL TEN: {order.sell_first.mul(order.turns).sum():,.0f} of "
      f"{order.turns.sum():,} turns place the sell first. "
      f"Exceptions: {order.exceptions.sum()}.")

In [ ]:
d = order.sort_values("turns")
y = np.arange(len(d))
fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.barh(y, d.sell_first, height=0.55, color=SELL, zorder=3,
        label="observed: sell before the purchase")
ax.scatter(d.shuffled, y, marker="|", s=280, linewidths=2.4, color=ACC, zorder=4,
           label="same orders, shuffled")
for yy, (v, nn) in enumerate(zip(d.sell_first, d.turns)):
    ax.text(v + 0.012, yy, f"{v:.1%}  (n={nn:,})", va="center", fontsize=8.5,
            color="#334155")
ax.set_yticks(y); ax.set_yticklabels([label(s) for s in d.submission], fontsize=8.5)
ax.set_xlim(0, 1.22); ax.set_xticks(np.arange(0, 1.01, 0.25))
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}" if v <= 1 else "")
ax.set_xlabel("share of mixed turns in which the SELL is ordered first")
ax.set_title("Ten teams, 9,208 opportunities, zero exceptions", loc="left")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

**9,208 of 9,208.** Not 99.9 %. Every structural purchase made by any of the ten highest-rated submissions, in every episode in this corpus, is preceded in its own order list by a sale.

The shuffled baseline sits at 70 %, which is where a list containing several sells and one purchase lands by chance. The observed value is 30 points above it with zero variance.

**So the sell schedule of a strong route is not only a trading policy. It is the mechanism by which the farm is paid for**, and it is expressed in a channel. The order of a list, that no similarity measurement over action streams can see.

## 5. The market list is three channels wearing one name

The reason the previous notebook's map read this as free is that it treated the whole `market` list as one object. The engine does not. It accepts six operations there and they are three different kinds of thing.

| operation | what it actually does | can a mutant touch it? |
|---|---|---|
| `HIRE`, `BUY_LAND`, `BUY_ANIMAL` | changes the **farm** | no |
| `BUY_SEED`, `BUY_PRODUCT` | buys the **inputs** the plan consumes | only with a guarantee |
| `SELL` | trades | yes |

Hands are dismissed every night and re-hired every morning, so a hand that was never hired cannot execute the `hands` action recorded for it. A quadrant never bought cannot be planted. **Three of the six operations in the "market" channel are construction.**

In [ ]:
CLASS = {**{k: "structure" for k in ("HIRE", "BUY_LAND", "BUY_ANIMAL")},
         **{k: "inputs" for k in ("BUY_SEED", "BUY_PRODUCT")},
         "SELL": "trading"}
COLOR = {"structure": STRUCT, "inputs": INPUT, "trading": SELL}

# Per episode, how many of each operation. Then: across episodes of one agent,
# how many DISTINCT totals does each channel take? One means it never moves.
per_ep = {}
for (sub, eid), g in streams.groupby(["submission", "episode_id"]):
    c = {"structure": 0, "inputs": 0, "trading": 0}
    for s in g["market"]:
        for o in ops_of(s):
            k = CLASS.get(o)
            if k:
                c[k] += 1
    per_ep.setdefault(sub, []).append(c)

rows = []
for sub, eps in per_ep.items():
    if len(eps) < 5:
        continue
    r = {"submission": sub, "episodes": len(eps)}
    for k in ("structure", "inputs", "trading"):
        vals = [e[k] for e in eps]
        r[f"{k}_median"] = int(np.median(vals))
        r[f"{k}_distinct"] = len(set(vals))
    rows.append(r)
chan = pd.DataFrame(rows).sort_values("structure_distinct").reset_index(drop=True)
chan["agent"] = chan.submission.map(label)
display(chan[["agent", "episodes", "structure_median", "structure_distinct",
              "inputs_distinct", "trading_distinct"]])

In [ ]:
d = chan.sort_values("trading_distinct")
y = np.arange(len(d)); w = 0.26
fig, ax = plt.subplots(figsize=(8.4, 4.4))
for i, k in enumerate(("structure", "inputs", "trading")):
    ax.barh(y + (i - 1) * w, d[f"{k}_distinct"], height=w, color=COLOR[k],
            label=k, zorder=3)
ax.axvline(1, color="#0F172A", lw=1.2, ls=":", zorder=4)
ax.text(1.15, len(d) - 0.4, "1 = the total never changes,\nin any episode",
        fontsize=8.5, color="#334155", va="top")
ax.set_yticks(y); ax.set_yticklabels([label(s) for s in d.submission], fontsize=8.5)
ax.set_xlabel("distinct per-episode totals across this agent's episodes")
ax.set_title("The structural half of the market channel is as fixed as the board",
             loc="left")
ax.legend(frameon=False, fontsize=9, loc="lower right")
plt.tight_layout(); plt.show()

**Nine of the ten never vary their structural totals at all**: the same number of hires, the same two land purchases, the same nine animals, in every episode. The trading total varies for all of them.

That is the split the conservation map could not see, and it is why "the market channel is the mutable one" is true as a description and dangerous as a licence.

## 6. What this costs anyone replaying a route

A replayed route carries three things, and only the first is visible in a plot of its actions:

1. **a construction plan**, which tiles, in which order;
2. **an input schedule**. The seeds and goods the plan consumes, bought before they are needed;
3. **a cash-flow schedule**, sales sized and placed so that each purchase can afford itself on the turn it lands.

The third is expressed in *the order of a list*. It survives copying only if you copy the list verbatim, and it breaks silently the moment you regenerate the channel it lives in, which is exactly what an agent that "improves the trading" does.

**The recorded route's bank bottoms out at \$3.** It is timed to the dollar. That is not slack you can borrow against.

---

## 7. And a duel result that fell out of the same experiment

The first version of our seller held stock back whenever the price was poor. Measured against the parent over 20 recorded matchups, it **increased our own bank on the first game by \$5,354** and turned a \$7,964 win into a \$102,577 loss.

The reason is that both farms sell into one market whose price falls as inventory rises. Holding stock back leaves the high part of that curve **for the opponent**, whose recorded schedule sells into a market we politely declined to spoil. Pooled over the 20 matchups, the first seller lifted the *opponent's* median bank from \$80,421 to \$145,902.

Sweeping how aggressively to sell, the pattern is unambiguous:

| units per item per turn | our bank | **their** bank | margin |
|---|---|---|---|
| 3 | \$79,862 | \$88,844 | −6,934 |
| 40 | \$75,811 | \$77,078 | **−620** |

**Selling harder costs us about \$4,000 and costs the opponent about \$11,800.** A seller optimised on its own bank optimises the other farm's too. This is the practical content of *Dueling Algorithms* (Immorlica, Kalai, Lucier, Moitra, Postlewaite & Tennenholtz, STOC 2011) at the level of a single order: what is being maximised is P(beat the opponent), not E[value], and the two point in different directions here.

---

## 8. Addendum, 2026-08-26: the same lever measured at scale

*Added nine days after publication. Nothing above is retracted; this
section is evidence for the same claim on a larger sample, plus answers
to two of the open questions in section 9. The voice changes to first
person singular here because the addendum is written on its own.*

The original measurement rested on one matched pair of episodes and one
flag in one of my own agents. Since then I built a bit-exact C++ port of
the engine, which makes an episode cost under a millisecond, so the same
question can be asked of published agents directly rather than of a
harness I control.

### 8.1 Break the ordering on purpose, on three published agents

Take each agent as published, re-order every turn's market list, and
replay it against a strong adaptive public opponent at common random
seeds. Two orderings: a canonical sort, and a plain reversal as a
control, so that a result which depends on the specific permutation can
be told apart from one that depends on the ordering being destroyed at
all.

The three bars below are the margins those runs produced, measured on 2026-08-26 and recorded in my own run log as row 203bc. The chart draws them: it does not recompute them, and no episode is replayed inside this notebook.

In [ ]:
import matplotlib.pyplot as plt

# Measured margins from the 2026-08-26 reorder ablation, log row 203bc. They
# are data typed into this cell; the cell draws them and computes nothing.
ABLATION = [('shop-guard', 10637, 2855, 2842), ('morita tape', -9449, -92429, -61375), ('v21-r1', 1783, -88410, -60628)]

names = [a[0] for a in ABLATION]
pub = [a[1] / 1000 for a in ABLATION]
srt = [a[2] / 1000 for a in ABLATION]
rev = [a[3] / 1000 for a in ABLATION]
x = list(range(len(names)))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar([i - 0.26 for i in x], pub, width=0.24, color="#2f6fb2",
       label="as published")
ax.bar([i for i in x], srt, width=0.24, color="#c2571f",
       label="market list sorted")
ax.bar([i + 0.26 for i in x], rev, width=0.24, color="#9aa3ad",
       label="market list reversed (control)")
ax.axhline(0, color="#40484f", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("margin against the adaptive rival\n(thousands of $)")
ax.set_title("destroying the order of the market list, three published agents",
             fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="lower left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**Two different things are in that chart, and reading them as one
would be the same class of error the notebook is about.**

The 60,000 to 90,000 collapses on the second and third agents are the
financing chain of section 2, now measured on agents that are not mine.
A canonical sort places `BUY_` before `SELL`, the engine settles index 0
first, and a purchase whose funding sale now lands after it fails in
silence. The original figure of about 95,000 a game came from one flag
in one agent; here it reproduces on two more, from two different teams.

The first agent loses only 7,782, and the reason is instructive: it
already buys before it sells, so it never leans on same-turn financing
and has no chain to break. What it loses instead is a pure **execution**
effect, which of its units meet the better price. That agent is the
top-ranked one in this comparison, which makes the number worth
knowing.

### 8.2 The mechanism, stated more precisely than I could in August

Two readings of the interpreter were tested before either was used, and
the first one was wrong.

**A failed order does not abort the rest of the list.**
`_process_market` iterates over order INDICES with `for i in
range(max_len)` and runs the per-unit lockstep inside each index, so a
commit that fails ends that index's order and index `i + 1` starts
clean. I had assumed otherwise, and the assumption would have made the
rest of this section wrong.

**The market list is a positional contest.** Your order at index `i` is
quoted against the opponent's order at index `i`, both against the same
pre-commit inventory, and both advance one unit per iteration of the
loop. So the index a good sits at decides which of your units clear
before the opponent's supply has moved the price.

That is measurable rather than rhetorical. Counting my own sell units by
whether the opponent held the same good at the same index, over a full
episode:

*Counted on the morita tape at seed 11, measured on 2026-08-26 and recorded in my own run log as row 203bd. These are the counts that run produced, written out here rather than recomputed: the episode is not replayed in this notebook.*

| good | head to head | uncontested | contested share |
|---|---:|---:|---:|
| FERTILIZER | 1,631 | 275 | 85.6% |
| WHEAT | 337 | 630 | 34.9% |
| MILK | 187 | 141 | 57.0% |
| STRAWBERRY | 105 | 149 | 41.3% |
| MELON | 60 | 60 | 50.0% |
| WOOL | 24 | 127 | 15.9% |
| CARROT | 0 | 15 | 0.0% |
| **TOTAL** | **2,344** | **1,397** | **62.7%** |

**Nearly two thirds of my sell units meet the opponent's supply of
the same good at the same index**, and fertilizer carries most of it.
Both agents run short market lists, zero to three orders on most turns,
which is exactly why they collide: there are few indices to collide in.

### 8.3 So can the ordering be optimised? No, and the negative is
robust

If order decides which units clear first, the obvious move is to sort
the sells so the contested goods go first. It does not work, and I want
to publish that clearly, because a negative result of this shape saves
somebody a day.

Nine rules, each applied only to sells that are free to move, with
sells kept ahead of purchases so the financing chain is never the thing
being tested. Three tapes, common random seeds, against the same
adaptive opponent. The number is the change in margin against leaving
the recorded order alone.

*Nine rules plus the control, on three recorded tapes at common random seeds, measured on 2026-08-26 and recorded in my own run log as row 203bd. A dash means the rule was not run on that tape. Written out here rather than recomputed: no episode is replayed in this notebook.*

| ordering rule | tape A | tape B | tape C |
|---|---:|---:|---:|
| collapse severity first | -1,119 | -272 | - |
| collapse severity last | -2,781 | -404 | - |
| unit price first | -905 | -290 | - |
| unit price last | -3,407 | -386 | - |
| book depression first | -2,525 | -278 | - |
| book depression last | -2,486 | -384 | - |
| contested volume first | -3,196 | -439 | -3,974 |
| contested volume last | -1,761 | -132 | -826 |
| plain reversal (control) | -3,393 | -596 | - |
| recorded order kept | -7 | +25 | +0 |

**Every rule loses.** Collapse severity, unit price, book
depression, contested volume, each of them and each of their inverses,
and a plain reversal as a control: all negative, on all three tapes. The
only variant that breaks even is keeping the recorded order, and every
tape already satisfies it.

The reading I take from that is the one the notebook has been making
throughout, now with a sharper edge. **The order of the market list is
real and already claimed.** It is worth tens of thousands to break and
nothing to re-optimise, which makes it a constraint to preserve rather
than a free parameter to tune. Any mutation operator that reorders sells
should have to earn its place against that control before it is
allowed near a submission.

### 8.4 Two of the open questions in section 9, answered

**"How much of the field's apparent adaptivity is financing?"** I can
now answer this for the strongest agent in the comparison, by driving it
against an empty market and against a live opponent at the same seed and
diffing the two emitted streams turn by turn. It differs on 46 turns.
Thirty-nine of those are market-channel differences, and **all
thirty-nine are pure reorderings**: identical orders, identical goods,
identical quantities, in a different sequence.

```
turn 266   vs empty market : [BUY_PRODUCT WHEAT 5] [SELL FERTILIZER 1] [SELL FERTILIZER 3]
           vs live opponent: [BUY_PRODUCT WHEAT 5] [SELL FERTILIZER 3] [SELL FERTILIZER 1]
```

So for that agent the reactive part is not a budget and not a trader
either. It is a **sequencer**. That is a third answer the original
section 9 did not offer as an option.

**"What does a correct mutant look like?"** For the ordering half, on
the evidence in 8.3, it looks like the recorded one. Regenerating the
channel means reproducing an ordering that nine reasonable rules fail to
match, which is a much stronger constraint than the original section 6
could state.

**"Is the ordering deliberate or emergent?"** Still open, and I now
think it may not matter. The ablation prices the lever whether the agent
means it or not.

## 9. What we would tell ourselves before starting

**A conservation map says where variation is tolerated. It does not say where change is safe.** Those differ whenever the varying region carries a function that is not the one you are looking at. In this game the sells vary, and they also pay the bills.

The general form, and it is not specific to this competition: **an instrument can measure something true and adjacent to what you need.** That failure survives code review, produces plausible numbers, and is only caught by carrying a control that says what the measurement looks like when nothing is happening. Ours are a bit-identical parent (4,314 turns, 0 differences) and a within-agent band (two episodes of the same agent agree on 94.7 % of the stream and 0 tiles of the board).

**And the honest bottom line: the knockout is still worse than the route it replaces**, 7 of 20 on held-out seats against 19 of 20. We are publishing the mechanism, not a better agent. The mechanism is the part that generalises.

## 10. Open questions

*Section 8.4 now answers the second and third of these, and sharpens the first.*

**Is the ordering deliberate or emergent?** Nothing here separates an agent that reasons about cash from one whose template happens to emit sells first. A bank trace per turn would settle it and the replays contain one.

**How much of the field's apparent adaptivity is financing?** If sell *quantities* are set by what the next purchase costs rather than by what the market is paying, then a large part of the "reactive seller" everyone including us has described is a budget, not a trader.

**What does a correct mutant look like?** One that regenerates the trading channel while reproducing the cash-flow constraint it carries, which means solving for the schedule instead of copying it. That is an optimal-execution problem with a solvency constraint, and it is where we are now.

---

*Corpus: [kaggriculture top-10 replay streams](https://www.kaggle.com/datasets/destbreso/kaggriculture-top10-replay-streams), 1,482 episodes of the ten highest-rated submissions, taken from the competition's own episode API. The knockout agent replays another competitor's publicly observable route with credit; the seller is ours.*